# Modified GBM — model specification

**This notebook is the documentation source of truth.** The PDF `V3_modified_gbm_model.pdf` in this folder is the same text.

Modified GBM is a discrete-time replacement for geometric Brownian motion used in the V3 1.5-year monthly 10,000-path study.

## 1. Idea

Standard GBM draws each log-return from one normal \(N(\mu\Delta t,\sigma^2\Delta t)\). Signs are independent and up/down sizes share one volatility.

Modified GBM keeps the price map \(S_{t+1}=S_t e^{r_t}\) but splits \(r_t\) into:

1. **Direction** — two-state Markov chain on \(\{U,D\}\).
2. **Magnitude** — \(|N(\mu_U,\sigma_U^2)|\) on up bars and \(|N(\mu_D,\sigma_D^2)|\) on down bars.
3. **Price** — \(r_t=+m_t\) or \(r_t=-m_t\), then \(S\leftarrow S e^{r_t}\).

It is return-based. Option quotes are not used in calibration. American calls are priced by LSM on risk-neutral paths.

It is **not** Heston (no variance diffusion), **not** Merton (no jumps), **not** GARCH (no \(\omega,\alpha,\beta\)), and **not** hidden-state regime-switching (the state is the previous observed sign).

## 2. Estimation

On each lookback window of log-returns \(R_s=\ln(S_s/S_{s-1})\):

- Drop zeros. Count consecutive sign pairs. Laplace-smoothed
  \(\hat P(U\mid U)=(n_{UU}+1/2)/(n_{\mathrm{from\ }U}+1)\), and the analogue for \(\hat P(D\mid D)\).
- \(\mu_U,\sigma_U\) (resp. \(\mu_D,\sigma_D\)) = mean and sample SD of \(|R|\) on up (resp. down) bars.
- `last_up` = sign of the last non-zero lookback return (starts the simulator).

Rolling: 18-month window ending at each month-end (plus period start). Parameters dated \(t\) are used only after \(t\).

## 3. Simulation

**P-measure** (stock PDF): no drift shift. Path cloud \(n=10000\), seed 42. Reported path = p50 vs realized adj-close.

**Q-measure** (decision / moneyness PDFs): after drawing \(r\), shift
\(r\leftarrow r+(r_f\Delta t-\log\mathbb{E}e^{r})\) so \(\mathbb{E}e^{r}=e^{r_f\Delta t}\), \(\Delta t=1/252\). Then Longstaff–Schwartz on the same Monday ATM listed-call sample as every other model.

## 4. Parameters in the estimation PDF

\(P(U\mid U),\ P(D\mid D),\ P(U\mid D),\ P(D\mid U),\ \mu_U,\ \sigma_U,\ \mu_D,\ \sigma_D\).

## 5. Where results are

- Seven-model ranking: `V3_1p5y_monthly_empirical_study.pdf`
- Return-based group (includes Modified GBM): `V3_1p5y_monthly_empirical_study_return_based.pdf`
- Parameters / stock / moneyness: the matching `V3_1p5y_monthly_*.pdf` files in this folder
- Code: `V3-Models_result/modified gbm notebook/20*_modified_gbm.ipynb`
